# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this one over freestyle because notebook 01 already showed me there's real, learnable signal in this exact task — the lane guide's own numbers show a random forest hitting Precision@50 of 0.740 against a hand-written stale×visible rule's 0.240 on this same starter data. That's not a hypothetical; I've watched it happen. It also maps onto something a real content team would use every sprint — a ranked list of pages to look at first — rather than a one-off chart, which makes the 7 weeks feel worth spending here.

In [1]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages across {df['client_id'].nunique()} clients — the starter slice I'm validating my lane choice against.")

30,000 pages across 32 clients — the starter slice I'm validating my lane choice against.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** which pages should a content editor pull off the backlog and review first this sprint, given a whole client's worth of tracked pages and limited hours.

**Who acts, and how:** a content editor (or whoever owns the refresh backlog) works down my ranked queue and decides, per page, whether to refresh, expand, protect, or leave it alone.

**Cost of a wrong call — and it's not symmetric:** flagging a page that's actually fine costs an editor an hour on the wrong page — annoying but cheap. Missing a page that's genuinely declining, with real demand behind it, costs ongoing traffic that keeps compounding the longer it goes unreviewed — more expensive. That asymmetry is why I care about Precision@K at the *top* of the queue, not overall accuracy: a model that's mediocre everywhere but excellent in its top 20-50 is exactly what an editor with limited capacity needs.

In [2]:
# Sizing the cost side of the decision: how many pages are in the highest-value-to-protect
# bucket, where a missed decline is most expensive (they already earn strong position)?
decay_risk = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
print(f"{decay_risk.sum():,} pages ({decay_risk.mean()*100:.1f}%) are aged page-one pages "
      "(position 1-10, 180+ days old) — exactly the pages where a missed decline is expensive, "
      "since they already earn strong position and demand.")

7,076 pages (23.6%) are aged page-one pages (position 1-10, 180+ days old) — exactly the pages where a missed decline is expensive, since they already earn strong position and demand.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
down_share = (df["trend_direction"].str.lower() == "down").mean() * 100
print(f"1) {down_share:.1f}% of all 30,000 pages are currently trending down — too large a group to "
      "'review everything,' which is exactly why prioritization is the actual problem, not just modeling.")

decay_risk = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
decay_declining_share = (decay_risk & (df["trend_direction"].str.lower() == "down")).sum() / decay_risk.sum() * 100
print(f"\n2) {decay_risk.sum():,} pages ({decay_risk.mean()*100:.1f}%) are aged page-one pages "
      f"(position 1-10, 180+ days old). {decay_declining_share:.1f}% of those are already declining — "
      "these are high-value pages worth protecting first.")

wc_by_trend = df.groupby("trend_direction")["word_count"].median()
print(f"\n3) Median word count, down vs up pages: {wc_by_trend['down']:.0f} vs {wc_by_trend['up']:.0f} — "
      "almost identical. Word count alone won't separate decliners from growers (I found the same thing "
      "in notebook 01's Discovery C), so a useful score needs more signal than just content length.")

1) 54.2% of all 30,000 pages are currently trending down — too large a group to 'review everything,' which is exactly why prioritization is the actual problem, not just modeling.

2) 7,076 pages (23.6%) are aged page-one pages (position 1-10, 180+ days old). 51.8% of those are already declining — these are high-value pages worth protecting first.

3) Median word count, down vs up pages: 2909 vs 2848 — almost identical. Word count alone won't separate decliners from growers (I found the same thing in notebook 01's Discovery C), so a useful score needs more signal than just content length.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work will be able to say:** observed and directional patterns in this dataset (e.g. 'aged page-one pages are declining at a notably higher rate than the base rate'), and decision-support output — a ranked queue an editor can use to prioritize limited review time, with reason codes they can inspect and override.

**What it will never say:** that a refresh *causes* recovery (that needs an actual experiment, not this data), that I've reverse-engineered any part of Google's ranking algorithm, or that a high score is a guarantee rather than a prioritization signal.

**One proxy-label caveat to carry forward:** the starter dataset's `trend_direction` is a snapshot bucket computed from the current window, not a future outcome — the lane guide flags this explicitly as a beginner proxy label. For the actual capstone I plan to move to a future-window label from the warehouse release (prior 90 days of features predicting the next 30 days), which is a fundamentally stronger claim than 'is this page currently down.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.